# S7 — AndinaLog 03B | Clasificación de desviación térmica

Se predice si habrá una desviación térmica durante los próximos 60 minutos. Se reutiliza el split temporal de S6 y se prioriza la detección de la clase positiva.

## 1. Contrato y datos

La clase positiva es una desviación futura. Una alerta permite revisar o intervenir el transporte. Un falso negativo deja una desviación sin advertencia; un falso positivo consume capacidad de inspección.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
SEMILLA = 42
TARGET = "clasificacion_objetivo_60min"
COSTO_FN = 5
COSTO_FP = 1

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv").is_file()), None)
    if raiz is None: raise FileNotFoundError("No se encontró la raíz del proyecto")
    return raiz

RAIZ = encontrar_raiz()
RUTA_DATOS = RAIZ / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv"
RUTA_SPLIT = RAIZ / "proyecto-integrador/04_regresion/salidas_s6/asignacion_split_viajes.csv"
SALIDAS = RAIZ / "proyecto-integrador/05_clasificacion/salidas_s7"
df = pd.read_csv(RUTA_DATOS, encoding="utf-8-sig")
split = pd.read_csv(RUTA_SPLIT, encoding="utf-8-sig")
df["timestamp_bolivia"] = pd.to_datetime(df["timestamp_bolivia"], errors="raise")
df["apta_clasificacion_60min"] = df["apta_clasificacion_60min"].astype("boolean")
modelado = df.loc[df["apta_clasificacion_60min"].fillna(False) & df[TARGET].isin([0, 1])].copy()
modelado[TARGET] = modelado[TARGET].astype(int)
modelado["particion"] = modelado["viaje_id"].map(split.set_index("viaje_id")["particion"])
assert len(df) == 28677 and len(modelado) == 28557
assert modelado["particion"].notna().all()
print("Lecturas aptas:", len(modelado), "| prevalencia:", round(modelado[TARGET].mean(), 4))


Lecturas aptas: 28557 | prevalencia: 0.0463


## 2. Variables y split congelado

Las variables futuras quedan excluidas. El mismo viaje nunca aparece en dos particiones y TEST conserva la separación de S6.

In [2]:
NUM_BASE = [
    "temp_c", "humedad_pct", "objetivo_c", "tolerancia_c", "desvio_respecto_umbral_c",
    "temp_lag1_c", "temp_lag2_c", "cambio_temp_c", "pendiente_c_por_min",
    "temp_media_historica_3", "temp_max_historica_3", "minutos_desde_inicio",
    "capacidad_kg_tratada", "cantidad_solicitada_tratado", "tiempo_entrega_prometido_hrs_tratado",
]
CAT_BASE = ["categoria_logistica_tratada", "tipo_camion_tratado", "centro_distribucion_tratado"]
NUM_EVENTOS = [
    "eventos_previos_60m", "eventos_previos_180m", "eventos_previos_24h",
    "alertas_temp_previas_60m", "alertas_temp_previas_180m", "alertas_temp_previas_24h",
    "fallas_motor_previas_24h", "eventos_alta_previos_24h",
    "eventos_no_reconocidos_previos_24h", "reconocimiento_faltante_previos_24h",
    "minutos_desde_ultimo_evento", "minutos_desde_ultima_alerta_temp",
    "umbral_eventos_temp_cabina_c", "geocerca_eventos_radio_km",
    "dias_desde_ultimo_mantenimiento_eventos",
]
FEATURES_BASE = NUM_BASE + CAT_BASE
FEATURES_EVENTOS = NUM_BASE + NUM_EVENTOS + CAT_BASE
PROHIBIDAS = {TARGET, "max_desvio_termico_proximos_60min_c", "n_lecturas_futuras_60min",
              "apta_clasificacion_60min", "apta_regresion_60min"}
assert not set(FEATURES_EVENTOS) & PROHIBIDAS

idx_aj = modelado.index[modelado["particion"].eq("AJUSTE")]
idx_val = modelado.index[modelado["particion"].eq("VALIDACION")]
idx_test = modelado.index[modelado["particion"].eq("TEST")]
idx_pretest = modelado.index[modelado["particion"].isin(["AJUSTE", "VALIDACION"])]
assert min(len(idx_aj), len(idx_val), len(idx_test)) > 0
assert set(modelado.loc[idx_aj, "viaje_id"]).isdisjoint(modelado.loc[idx_val, "viaje_id"])
assert set(modelado.loc[idx_pretest, "viaje_id"]).isdisjoint(modelado.loc[idx_test, "viaje_id"])
resumen_split = modelado.loc[modelado["particion"].isin(["AJUSTE", "VALIDACION", "TEST"])].groupby("particion").agg(
    lecturas=(TARGET, "size"), positivos=(TARGET, "sum"), prevalencia=(TARGET, "mean"), viajes=("viaje_id", "nunique"))
print(resumen_split.to_string())


            lecturas  positivos  prevalencia  viajes
particion                                           
AJUSTE         18946        876     0.046237     796
TEST            4274        232     0.054282     180
VALIDACION      4287        151     0.035223     180


## 3. Modelos y selección

Se comparan Dummy, regresión logística y Random Forest. Las versiones base y con eventos se seleccionan por PR-AUC de validación, adecuada para una clase positiva poco frecuente.

In [3]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve)

def preprocesador(numericas, categoricas, escalar=True):
    pasos = [("imputar", SimpleImputer(strategy="median", add_indicator=True))]
    if escalar: pasos.append(("escalar", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(pasos), numericas),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoricas)])

def crear_modelos():
    modelos = {"Dummy mayoritaria": (FEATURES_BASE, DummyClassifier(strategy="most_frequent"))}
    for etiqueta, nums, feats in [("Base", NUM_BASE, FEATURES_BASE),
                                  ("Con eventos", NUM_BASE + NUM_EVENTOS, FEATURES_EVENTOS)]:
        for c in [0.05, 0.2, 1.0, 5.0]:
            for peso in [None, "balanced"]:
                nombre = f"Logística {etiqueta} C={c} peso={peso or 'ninguno'}"
                modelos[nombre] = (feats, Pipeline([
                    ("pre", preprocesador(nums, CAT_BASE, True)),
                    ("modelo", LogisticRegression(C=c, class_weight=peso, max_iter=2000, random_state=SEMILLA))]))
        modelos[f"Random Forest {etiqueta}"] = (feats, Pipeline([
            ("pre", preprocesador(nums, CAT_BASE, False)),
            ("modelo", RandomForestClassifier(n_estimators=180, max_depth=10, min_samples_leaf=5,
                                               class_weight="balanced_subsample", random_state=SEMILLA, n_jobs=-1))]))
    return modelos

def metricas_prob(real, prob, umbral=.5):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(real, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(real, pred), "balanced_accuracy": balanced_accuracy_score(real, pred),
            "precision": precision_score(real, pred, zero_division=0), "recall": recall_score(real, pred, zero_division=0),
            "f1": f1_score(real, pred, zero_division=0), "pr_auc": average_precision_score(real, prob),
            "roc_auc": roc_auc_score(real, prob), "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "alertas": int(pred.sum()), "tasa_alertas": float(pred.mean())}

modelos = crear_modelos()
y = modelado[TARGET]
filas_val, probs_val = [], {}
for nombre, (feats, modelo) in modelos.items():
    modelo.fit(modelado.loc[idx_aj, feats], y.loc[idx_aj])
    if hasattr(modelo, "predict_proba"):
        prob = modelo.predict_proba(modelado.loc[idx_val, feats])[:, 1]
    else:
        prob = modelo.predict(modelado.loc[idx_val, feats]).astype(float)
    probs_val[nombre] = prob
    filas_val.append({"modelo": nombre, "familia_variables": "eventos" if "Con eventos" in nombre else "base",
                      **metricas_prob(y.loc[idx_val], prob, .5)})
metricas_val = pd.DataFrame(filas_val).sort_values(["pr_auc", "recall"], ascending=False)
seleccion = metricas_val.iloc[0]["modelo"]
print(metricas_val.head(10).round(4).to_string(index=False))
print("Modelo seleccionado por PR-AUC de validación:", seleccion)


c:\Users\remrodri\Github\practicasNotebookColab\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\remrodri\Github\practicasNotebookColab\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\remrodri\Github\practicasNotebookColab\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\remrodri\Github\practicasNotebookColab\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\remrodri\Github\practicasNotebookColab\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\remrodri\Github\practicasNotebookColab\.v

                                  modelo familia_variables  accuracy  balanced_accuracy  precision  recall     f1  pr_auc  roc_auc   TN  FP  FN  TP  alertas  tasa_alertas
                      Random Forest Base              base    0.9519             0.6688     0.3333  0.3642 0.3481  0.3981   0.7635 4026 110  96  55      165        0.0385
               Random Forest Con eventos           eventos    0.9461             0.6690     0.2917  0.3709 0.3265  0.3900   0.7600 4000 136  95  56      192        0.0448
       Logística Base C=1.0 peso=ninguno              base    0.9720             0.6346     0.8039  0.2715 0.4059  0.3608   0.7439 4126  10 110  41       51        0.0119
       Logística Base C=0.2 peso=ninguno              base    0.9718             0.6312     0.8000  0.2649 0.3980  0.3595   0.7456 4126  10 111  40       50        0.0117
       Logística Base C=5.0 peso=ninguno              base    0.9720             0.6346     0.8039  0.2715 0.4059  0.3593   0.7421 4126  10 110  

## 4. Umbral operativo

El umbral se elige en validación con un costo didáctico de 5 por falso negativo y 1 por falso positivo. El equipo operativo deberá confirmar esta relación.

In [4]:
# Selección del umbral solo en VALIDACIÓN. Relación de costos didáctica: FN = 5, FP = 1.
prob_val = probs_val[seleccion]
filas_umbral = []
for u in np.round(np.arange(.01, 1.00, .01), 2):
    m = metricas_prob(y.loc[idx_val], prob_val, u)
    filas_umbral.append({"umbral": u, **m, "costo_relativo": COSTO_FN*m["FN"] + COSTO_FP*m["FP"]})
tabla_umbral = pd.DataFrame(filas_umbral)
mejor = tabla_umbral.sort_values(["costo_relativo", "FN", "alertas", "umbral"], ascending=[True, True, True, False]).iloc[0]
UMBRAL = float(mejor["umbral"])
print("Umbral seleccionado:", UMBRAL)
print(mejor[["TN", "FP", "FN", "TP", "precision", "recall", "f1", "pr_auc", "alertas", "tasa_alertas", "costo_relativo"]].to_string())
print("La relación 5:1 es un supuesto didáctico que el negocio debe confirmar.")


Umbral seleccionado: 0.7
TN                4121.000000
FP                  15.000000
FN                 100.000000
TP                  51.000000
precision            0.772727
recall               0.337748
f1                   0.470046
pr_auc               0.398052
alertas             66.000000
tasa_alertas         0.015395
costo_relativo     515.000000
La relación 5:1 es un supuesto didáctico que el negocio debe confirmar.


## 5. Test final

Modelo y umbral quedan fijados antes de abrir TEST. La matriz de confusión muestra explícitamente falsos negativos y falsos positivos.

In [5]:
# Reajuste en PRETEST y apertura única de TEST.
feats_sel, modelo_final = crear_modelos()[seleccion]
modelo_final.fit(modelado.loc[idx_pretest, feats_sel], y.loc[idx_pretest])
prob_test = modelo_final.predict_proba(modelado.loc[idx_test, feats_sel])[:, 1]
metricas_test = metricas_prob(y.loc[idx_test], prob_test, UMBRAL)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(modelado.loc[idx_pretest, FEATURES_BASE], y.loc[idx_pretest])
prob_dummy = dummy.predict_proba(modelado.loc[idx_test, FEATURES_BASE])[:, 1]
metricas_dummy = metricas_prob(y.loc[idx_test], prob_dummy, .5)
print("TEST modelo seleccionado:", metricas_test)
print("TEST baseline:", metricas_dummy)


TEST modelo seleccionado: {'accuracy': 0.9635002339728591, 'balanced_accuracy': np.float64(0.7227057704448122), 'precision': 0.7835820895522388, 'recall': 0.4525862068965517, 'f1': 0.5737704918032787, 'pr_auc': np.float64(0.5255511736962303), 'roc_auc': np.float64(0.843713209575321), 'TN': np.int64(4013), 'FP': np.int64(29), 'FN': np.int64(127), 'TP': np.int64(105), 'alertas': 134, 'tasa_alertas': 0.0313523631258774}
TEST baseline: {'accuracy': 0.9457182966775854, 'balanced_accuracy': np.float64(0.5), 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'pr_auc': np.float64(0.0542817033224146), 'roc_auc': np.float64(0.5), 'TN': np.int64(4042), 'FP': np.int64(0), 'FN': np.int64(232), 'TP': np.int64(0), 'alertas': 0, 'tasa_alertas': 0.0}


## 6. Evidencias

Se exportan métricas, sensibilidad del umbral y predicciones de test para interpretar el costo y la carga de alertas.

In [6]:
SALIDAS.mkdir(parents=True, exist_ok=True)
metricas_val.to_csv(SALIDAS / "metricas_validacion_clasificacion_s7.csv", index=False, encoding="utf-8-sig")
tabla_umbral.to_csv(SALIDAS / "sensibilidad_umbral_validacion_s7.csv", index=False, encoding="utf-8-sig")

resumen_test = pd.DataFrame([
    {"modelo": seleccion, "umbral": UMBRAL, **metricas_test},
    {"modelo": "Dummy mayoritaria", "umbral": .5, **metricas_dummy},
])
resumen_test.to_csv(SALIDAS / "metricas_test_clasificacion_s7.csv", index=False, encoding="utf-8-sig")

pred = modelado.loc[idx_test, ["fila_bronze", "viaje_id", "timestamp_bolivia", TARGET,
                               "categoria_logistica_tratada", "tipo_camion_tratado",
                               "centro_distribucion_tratado", "tiene_evento_previo_24h"]].copy()
pred["probabilidad_desviacion_60min"] = prob_test
pred["umbral_seleccionado"] = UMBRAL
pred["prediccion"] = (prob_test >= UMBRAL).astype(int)
pred["tipo_resultado"] = np.select([
    pred[TARGET].eq(1) & pred["prediccion"].eq(1), pred[TARGET].eq(0) & pred["prediccion"].eq(0),
    pred[TARGET].eq(0) & pred["prediccion"].eq(1)], ["TP", "TN", "FP"], default="FN")
pred["timestamp_bolivia"] = pred["timestamp_bolivia"].dt.strftime("%Y-%m-%d %H:%M:%S")
pred.to_csv(SALIDAS / "predicciones_test_clasificacion_s7.csv", index=False, encoding="utf-8-sig")

precision, recall, _ = precision_recall_curve(y.loc[idx_test], prob_test)
cm = confusion_matrix(y.loc[idx_test], pred["prediccion"], labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(recall, precision)
axes[0].axhline(y.loc[idx_test].mean(), linestyle="--", color="gray", label="Prevalencia")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Curva precision-recall en TEST")
axes[0].legend()
im = axes[1].imshow(cm, cmap="Blues")
for (i, j), valor in np.ndenumerate(cm):
    axes[1].text(j, i, str(valor), ha="center", va="center",
                 color="white" if valor > cm.max()/2 else "black")
axes[1].set(xticks=[0,1], yticks=[0,1], xlabel="Predicho", ylabel="Real", title=f"Matriz de confusión (umbral {UMBRAL:.2f})")
plt.tight_layout()
plt.savefig(SALIDAS / "evaluacion_clasificacion_s7.png", dpi=150, bbox_inches="tight")
plt.close()
print("Salidas guardadas en", SALIDAS)


Salidas guardadas en c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\05_clasificacion\salidas_s7


## 7. Límite operativo

El resultado es una evaluación retrospectiva sobre datos sintéticos. La capacidad real para atender alertas y el costo relativo de los errores deben ser aprobados antes de usar el umbral en operación.